In [1]:
from langchain_community.graphs import Neo4jGraph

# Connect to Neo4j
graph = Neo4jGraph(
    url="neo4j+s://f5886a71.databases.neo4j.io",
    username="neo4j",
    password="fFCqS5-Ak3ZLppY0p1e2rrL1ZN_FJwt7oH_McxlCh5o"
)

print("✓ Connected to Neo4j")

/var/folders/_n/46drt9kd1_76dqk_vj0xk9qm0000gn/T/ipykernel_8662/1426366927.py:4: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


✓ Connected to Neo4j


In [ ]:
%store -r champion_full_data

# Then take the first 10:
# champions_to_process = champion_full_data[:10]

# For now, let's check if the variable exists
try:
    print(f"Champion data available: {len(champion_full_data)} champions")
    champions_to_process = champion_full_data
    for i, champ in enumerate(champions_to_process, 1):
        print(f"  {i}. {champ['name']}")
except NameError:
    print("⚠️ champion_full_data not found!")

Champion data available: 172 champions

Will process first 10 champions:
  1. Annie
  2. Olaf
  3. Galio
  4. Twisted Fate
  5. Xin Zhao
  6. Urgot
  7. LeBlanc
  8. Vladimir
  9. Fiddlesticks
  10. Kayle
  11. Master Yi
  12. Alistar
  13. Ryze
  14. Sion
  15. Sivir
  16. Soraka
  17. Teemo
  18. Tristana
  19. Warwick
  20. Nunu & Willump
  21. Miss Fortune
  22. Ashe
  23. Tryndamere
  24. Jax
  25. Morgana
  26. Zilean
  27. Singed
  28. Evelynn
  29. Twitch
  30. Karthus
  31. Cho'Gath
  32. Amumu
  33. Rammus
  34. Anivia
  35. Shaco
  36. Dr. Mundo
  37. Sona
  38. Kassadin
  39. Irelia
  40. Janna
  41. Gangplank
  42. Corki
  43. Karma
  44. Taric
  45. Veigar
  46. Trundle
  47. Swain
  48. Caitlyn
  49. Blitzcrank
  50. Malphite
  51. Katarina
  52. Nocturne
  53. Maokai
  54. Renekton
  55. Jarvan IV
  56. Elise
  57. Orianna
  58. Wukong
  59. Brand
  60. Lee Sin
  61. Vayne
  62. Rumble
  63. Cassiopeia
  64. Skarner
  65. Heimerdinger
  66. Nasus
  67. Nidalee
  68. Udy

In [4]:
# Clear everything
graph.query("""
MATCH (n)
DETACH DELETE n
""")

print("✓ Database cleared")

# Verify
result = graph.query("MATCH (n) RETURN count(n) as count")
print(f"Node count: {result[0]['count']}")

✓ Database cleared
Node count: 0


In [5]:
def create_champion(graph, champion_data):
    
    """
    Creates a Champion node with all its properties and role relationships.
    
    Args:
        graph: Neo4j graph connection
        champion_data: Dict with champion info from JSON
    
    Returns:
        Champion name (str)
    """
    # Extract basic info
    champ_id = champion_data['id']
    name = champion_data['name']
    title = champion_data['title']
    roles = champion_data.get('roles', [])  # Use .get() in case missing
    
    # Get tactical info (with defaults)
    tactical = champion_data.get('tacticalInfo', {})
    damage_type = tactical.get('damageType', 'unknown')
    attack_type = tactical.get('attackType', 'unknown')
    difficulty = tactical.get('difficulty', 0)
    
    # Create Champion node
    create_champ_query = """
    MERGE (c:Champion {id: $champion_id})
    SET c.name = $name,
        c.title = $title,
        c.damageType = $damageType,
        c.attackType = $attackType,
        c.difficulty = $difficulty
    RETURN c.name as name
    """
    
    graph.query(
        create_champ_query,
        params={
            'champion_id': champ_id,
            'name': name,
            'title': title,
            'damageType': damage_type,
            'attackType': attack_type,
            'difficulty': difficulty
        }
    )
    
    # Create Role relationships
    create_role_query = """
    MATCH (c:Champion {id: $champion_id})
    MERGE (r:Role {name: $role_name})
    MERGE (c)-[:HAS_ROLE]->(r)
    """
    
    for role in roles:
        graph.query(
            create_role_query,
            params={
                'champion_id': champ_id,
                'role_name': role.capitalize()
            }
        )
    
    return name


def create_abilities(graph, champion_data):
    """
    Creates Ability nodes and HAS_ABILITY relationships.
    
    Args:
        graph: Neo4j graph connection
        champion_data: Dict with champion info from JSON
    
    Returns:
        Number of abilities created (int)
    """
    champ_id = champion_data['id']
    abilities_created = 0
    
    # Create ability creation query
    create_ability_query = """
    MATCH (c:Champion {id: $champion_id})
    MERGE (a:Ability {id: $ability_id})
    SET a.name = $ability_name,
        a.slot = $slot,
        a.description = $description
    MERGE (c)-[:HAS_ABILITY]->(a)
    RETURN a.name as name
    """
    
    # 1. Process Passive
    passive = champion_data.get('passive')
    if passive:
        ability_id = f"{champ_id}_passive"
        graph.query(
            create_ability_query,
            params={
                'champion_id': champ_id,
                'ability_id': ability_id,
                'ability_name': passive.get('name', 'Unknown'),
                'slot': 'Passive',
                'description': passive.get('description', '')
            }
        )
        abilities_created += 1
    
    # 2. Process Spells (Q, W, E, R)
    spells = champion_data.get('spells', [])
    for spell in spells:
        spell_key = spell.get('spellKey', '').upper()  # 'q' -> 'Q'
        ability_id = f"{champ_id}_{spell_key.lower()}"
        
        graph.query(
            create_ability_query,
            params={
                'champion_id': champ_id,
                'ability_id': ability_id,
                'ability_name': spell.get('name', 'Unknown'),
                'slot': spell_key,
                'description': spell.get('description', '')
            }
        )
        abilities_created += 1
    
    return abilities_created

print("✓ Helper functions defined")

✓ Helper functions defined


In [6]:
# Process 10 champions
print("Starting ingestion...\n")

total_abilities = 0

for i, champ_data in enumerate(champions_to_process, 1):
    # Create champion and roles
    champ_name = create_champion(graph, champ_data)
    
    # Create abilities
    ability_count = create_abilities(graph, champ_data)
    total_abilities += ability_count
    
    print(f"✓ {i}/10: {champ_name} ({ability_count} abilities)")

print(f"\n🎉 Success! Created {len(champions_to_process)} champions with {total_abilities} total abilities")

Starting ingestion...

✓ 1/10: Annie (5 abilities)
✓ 2/10: Olaf (5 abilities)
✓ 3/10: Galio (5 abilities)
✓ 4/10: Twisted Fate (5 abilities)
✓ 5/10: Xin Zhao (5 abilities)
✓ 6/10: Urgot (5 abilities)
✓ 7/10: LeBlanc (5 abilities)
✓ 8/10: Vladimir (5 abilities)
✓ 9/10: Fiddlesticks (5 abilities)
✓ 10/10: Kayle (5 abilities)
✓ 11/10: Master Yi (5 abilities)
✓ 12/10: Alistar (5 abilities)
✓ 13/10: Ryze (5 abilities)
✓ 14/10: Sion (5 abilities)
✓ 15/10: Sivir (5 abilities)
✓ 16/10: Soraka (5 abilities)
✓ 17/10: Teemo (5 abilities)
✓ 18/10: Tristana (5 abilities)
✓ 19/10: Warwick (5 abilities)
✓ 20/10: Nunu & Willump (5 abilities)
✓ 21/10: Miss Fortune (5 abilities)
✓ 22/10: Ashe (5 abilities)
✓ 23/10: Tryndamere (5 abilities)
✓ 24/10: Jax (5 abilities)
✓ 25/10: Morgana (5 abilities)
✓ 26/10: Zilean (5 abilities)
✓ 27/10: Singed (5 abilities)
✓ 28/10: Evelynn (5 abilities)
✓ 29/10: Twitch (5 abilities)
✓ 30/10: Karthus (5 abilities)
✓ 31/10: Cho'Gath (5 abilities)
✓ 32/10: Amumu (5 abilitie

In [7]:
import re
def extract_damage_types(description):
    """
    Extract damage types from ability description.
    
    Returns: List of damage types found
    """
    damage_types = []
    text = description.lower()
    
    # Check for explicit damage type tags
    if '<magicdamage>' in text or 'magic damage' in text:
        damage_types.append('Magic')
    if '<physicaldamage>' in text or 'physical damage' in text:
        damage_types.append('Physical')
    if '<truedamage>' in text or 'true damage' in text:
        damage_types.append('True')
    
    return damage_types


def extract_crowd_control(description):
    """
    Extract crowd control effects from ability description.
    
    Returns: List of CC types found
    """
    cc_types = []
    text = description.lower()
    
    # CC keywords - order matters (check specific before general)
    cc_keywords = {
        'Stun': r'\bstun',
        'Root': r'\broot',
        'Knockup': r'\bknock(s|ing)?\s+up',
        'Knockback': r'\bknock(s|ing)?\s+(back|away)',
        'Taunt': r'\btaunt',
        'Charm': r'\bcharm',
        'Fear': r'\bfear',
        'Slow': r'\bslow',
        'Blind': r'\bblind',
        'Silence': r'\bsilence',
        'Grounded': r'\bground',
        'Disarm': r'\bdisarm',
        'Suppress': r'\bsuppress'
    }
    
    for cc_name, pattern in cc_keywords.items():
        if re.search(pattern, text):
            cc_types.append(cc_name)
    
    return cc_types


def extract_scaling_stats(description):
    """
    Extract what stats the ability scales with.
    
    Returns: List of stats found
    """
    stats = []
    text = description.lower()
    
    # Stat keywords
    stat_keywords = {
        'AP': r'\bability power|\bap\b',
        'AD': r'\battack damage|\bad\b|\bbad\b|bonus attack damage',
        'Health': r'\bmax(imum)?\s+health|\bbonushealth|\bbonus health',
        'Armor': r'\barmor',
        'MagicResist': r'\bmagic resist',
        'AttackSpeed': r'\battack speed',
        'CriticalStrike': r'\bcritical strike|\bcrit'
    }
    
    for stat_name, pattern in stat_keywords.items():
        if re.search(pattern, text):
            stats.append(stat_name)
    
    return stats


def extract_effect_types(description):
    """
    Extract effect types (utility, mobility, etc.).
    
    Returns: List of effects found
    """
    effects = []
    text = description.lower()
    
    # Effect keywords
    effect_keywords = {
        'Shield': r'\b<shield>|shield',
        'Heal': r'\bheal|\bhealing',
        'Dash': r'\bdash',
        'Blink': r'\bblink|\bteleport',
        'Stealth': r'\bstealth|\binvisib',
        'Movement': r'\bmove(ment)?\s+speed|\bms\b',
        'AttackSpeed': r'\battack\s+speed',
        'Lifesteal': r'\blifesteal|\blife\s+steal',
        'Vamp': r'\bspell\s+vamp'
    }
    
    for effect_name, pattern in effect_keywords.items():
        if re.search(pattern, text):
            effects.append(effect_name)
    
    return effects


print("✓ Parser functions defined")

# Test on Annie's passive
test_desc = "After casting 4 spells, Annie's next offensive spell will stun the target."
print(f"\nTest on: '{test_desc}'")
print(f"  CC found: {extract_crowd_control(test_desc)}")

✓ Parser functions defined

Test on: 'After casting 4 spells, Annie's next offensive spell will stun the target.'
  CC found: ['Stun']


In [8]:
# Add champion-level relationships
champion_query = """
MATCH (c:Champion {id: $champion_id})
MERGE (at:AttackType {name: $attack_type})
MERGE (c)-[:ATTACKS_WITH]->(at)
"""

print("Adding champion-level relationships...\n")

for champ in champions_to_process:
    tactical = champ.get('tacticalInfo', {})
    attack_type = tactical.get('attackType', 'unknown').capitalize()
    
    graph.query(
        champion_query,
        params={
            'champion_id': champ['id'],
            'attack_type': attack_type
        }
    )
    
    print(f"✓ {champ['name']}: {attack_type}")

print("\n✓ Champion relationships added")

Adding champion-level relationships...

✓ Annie: Ranged
✓ Olaf: Melee
✓ Galio: Melee
✓ Twisted Fate: Ranged
✓ Xin Zhao: Melee
✓ Urgot: Ranged
✓ LeBlanc: Ranged
✓ Vladimir: Ranged
✓ Fiddlesticks: Ranged
✓ Kayle: Melee
✓ Master Yi: Melee
✓ Alistar: Melee
✓ Ryze: Ranged
✓ Sion: Melee
✓ Sivir: Ranged
✓ Soraka: Ranged
✓ Teemo: Ranged
✓ Tristana: Ranged
✓ Warwick: Melee
✓ Nunu & Willump: Melee
✓ Miss Fortune: Ranged
✓ Ashe: Ranged
✓ Tryndamere: Melee
✓ Jax: Melee
✓ Morgana: Ranged
✓ Zilean: Ranged
✓ Singed: Melee
✓ Evelynn: Melee
✓ Twitch: Ranged
✓ Karthus: Ranged
✓ Cho'Gath: Melee
✓ Amumu: Melee
✓ Rammus: Melee
✓ Anivia: Ranged
✓ Shaco: Melee
✓ Dr. Mundo: Melee
✓ Sona: Ranged
✓ Kassadin: Melee
✓ Irelia: Melee
✓ Janna: Ranged
✓ Gangplank: Melee
✓ Corki: Ranged
✓ Karma: Ranged
✓ Taric: Melee
✓ Veigar: Ranged
✓ Trundle: Melee
✓ Swain: Ranged
✓ Caitlyn: Ranged
✓ Blitzcrank: Melee
✓ Malphite: Melee
✓ Katarina: Melee
✓ Nocturne: Melee
✓ Maokai: Melee
✓ Renekton: Melee
✓ Jarvan IV: Melee
✓ Elise: 

In [9]:
def enrich_ability(graph, champion_id, spell_data, slot):
    """
    Parse ability description and create relationships.
    """
    ability_id = f"{champion_id}_{slot.lower() if slot != 'Passive' else 'passive'}"
    
    # Get descriptions
    description = spell_data.get('description', '')
    dynamic_desc = spell_data.get('dynamicDescription', '')
    full_text = f"{description} {dynamic_desc}"
    
    # Extract all data
    damage_types = extract_damage_types(full_text)
    cc_types = extract_crowd_control(full_text)
    scaling_stats = extract_scaling_stats(full_text)
    effect_types = extract_effect_types(full_text)
    
    # Create damage type relationships
    for dmg_type in damage_types:
        query = """
        MATCH (a:Ability {id: $ability_id})
        MERGE (dt:DamageType {name: $damage_type})
        MERGE (a)-[:DEALS]->(dt)
        """
        graph.query(query, params={'ability_id': ability_id, 'damage_type': dmg_type})
    
    # Create CC relationships
    for cc in cc_types:
        query = """
        MATCH (a:Ability {id: $ability_id})
        MERGE (cc:CrowdControl {name: $cc_type})
        MERGE (a)-[:APPLIES]->(cc)
        """
        graph.query(query, params={'ability_id': ability_id, 'cc_type': cc})
    
    # Create scaling relationships
    for stat in scaling_stats:
        query = """
        MATCH (a:Ability {id: $ability_id})
        MERGE (s:Stat {name: $stat_name})
        MERGE (a)-[:SCALES_WITH]->(s)
        """
        graph.query(query, params={'ability_id': ability_id, 'stat_name': stat})
    
    # Create effect relationships
    for effect in effect_types:
        query = """
        MATCH (a:Ability {id: $ability_id})
        MERGE (e:EffectType {name: $effect_name})
        MERGE (a)-[:HAS_EFFECT]->(e)
        """
        graph.query(query, params={'ability_id': ability_id, 'effect_name': effect})
    
    return {
        'damage': damage_types,
        'cc': cc_types,
        'scaling': scaling_stats,
        'effects': effect_types
    }

print("Enriching abilities...\n")

total_relationships = 0

for champ in champions_to_process:
    champ_id = champ['id']
    champ_name = champ['name']
    
    print(f"\n{champ_name}:")
    
    # Process passive
    passive = champ.get('passive')
    if passive:
        result = enrich_ability(graph, champ_id, passive, 'Passive')
        rel_count = sum(len(v) for v in result.values())
        total_relationships += rel_count
        print(f"  [P] {passive.get('name')}: {rel_count} relationships")
        if result['cc']:
            print(f"      CC: {', '.join(result['cc'])}")
    
    # Process spells
    spells = champ.get('spells', [])
    for spell in spells:
        slot = spell.get('spellKey', '').upper()
        result = enrich_ability(graph, champ_id, spell, slot)
        rel_count = sum(len(v) for v in result.values())
        total_relationships += rel_count
        
        # Print interesting info
        info_parts = []
        if result['damage']:
            info_parts.append(f"Dmg: {','.join(result['damage'])}")
        if result['cc']:
            info_parts.append(f"CC: {','.join(result['cc'])}")
        if result['effects']:
            info_parts.append(f"FX: {','.join(result['effects'][:2])}")  # First 2 effects
        
        info_str = ' | '.join(info_parts) if info_parts else 'no special effects'
        print(f"  [{slot}] {spell.get('name')}: {info_str}")

print(f"\n\n🎉 Created {total_relationships} ability relationships!")

Enriching abilities...


Annie:
  [P] Pyromania: 1 relationships
      CC: Stun
  [Q] Disintegrate: Dmg: Magic
  [W] Incinerate: Dmg: Magic
  [E] Molten Shield: Dmg: Magic | FX: Shield,Movement
  [R] Summon: Tibbers: Dmg: Magic | CC: Stun | FX: Movement,AttackSpeed

Olaf:
  [P] Berserker Rage: 4 relationships
  [Q] Undertow: Dmg: Physical | CC: Slow,Grounded | FX: Movement
  [W] Tough It Out: FX: Shield,Heal
  [E] Reckless Swing: Dmg: True | FX: Heal
  [R] Ragnarok: FX: Movement

Galio:
  [P] Colossal Smash: 1 relationships
  [Q] Winds of War: Dmg: Magic | FX: Heal
  [W] Shield of Durand: Dmg: Magic,Physical | CC: Taunt,Slow | FX: Shield
  [E] Justice Punch: Dmg: Magic | CC: Knockup | FX: Dash
  [R] Hero's Entrance: Dmg: Magic | CC: Stun,Knockup | FX: Shield

Twisted Fate:
  [P] Loaded Dice: 0 relationships
  [Q] Wild Cards: Dmg: Magic
  [W] Pick a Card: Dmg: Magic | CC: Stun,Slow
  [E] Stacked Deck: Dmg: Magic | FX: AttackSpeed
  [R] Destiny: FX: Blink

Xin Zhao:
  [P] Determination: 

In [10]:
# Query 3: Find all abilities that scale with AP
query = """
MATCH (c:Champion)-[:HAS_ABILITY]->(a:Ability)
MATCH (a)-[:SCALES_WITH]->(s:Stat {name: 'AP'})
RETURN c.name as champion, count(a) as ap_abilities
ORDER BY ap_abilities DESC
LIMIT 5
"""

results = graph.query(query)
print("\nChampions with Most AP Scaling Abilities:")
for row in results:
    print(f"  {row['champion']}: {row['ap_abilities']} abilities")

[#C464]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv4Address(('34.126.64.110', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): TimeoutError(60, 'Operation timed out')
Unable to retrieve routing information
Transaction failed and will be retried in 1.0291981446890288s (Unable to retrieve routing information)
[#C43A]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('si-f5886a71-8c08.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))): TimeoutError(60, 'Operation timed out')
Transaction failed and will be retried in 2.2128093424380326s (Failed to read from defunct connection IPv4Address(('si-f5886a71-8c08.production-orch-0064.neo4j.io', 7687)) (ResolvedIPv4Address(('34.126.64.110', 7687))))



Champions with Most AP Scaling Abilities:
  Veigar: 2 abilities
  Singed: 1 abilities
  Vladimir: 1 abilities
  Akali: 1 abilities
  Ryze: 1 abilities
